# NBA Analytics Database

This notebook creates a small NBA database and runs queries for player performance, team statistics, injuries, and advanced metrics.

## API Import Option

The notebook below uses the local CSV files so it can run offline. The project also includes `api_import.py`, which rebuilds the same SQLite database using live NBA API calls through the `nba_api` package. From the project folder, run:

```bash
python api_import.py --season 2023-24 --max-games 5
```

That script loads teams, games, players, traditional box scores, and advanced metrics from the API. Injury data stays as a local/manual table because `nba_api` does not provide a simple official injury-report endpoint.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## Create the Database

The setup script creates the tables and loads the CSV files in the `data` folder.

In [ ]:
project_dir = Path.cwd().parent
sys.path.append(str(project_dir))

from database_setup import create_database, load_all_data

database_path = project_dir / "nba_analytics.db"
connection = create_database(database_path)
loaded_counts = load_all_data(connection)

pd.Series(loaded_counts, name="Rows Loaded")

## View the Player Data

In [ ]:
players = pd.read_sql_query(
    """
    SELECT
        p.player_id,
        p.first_name || ' ' || p.last_name AS player,
        p.position,
        t.abbreviation AS team
    FROM players p
    JOIN teams t ON p.team_id = t.team_id
    ORDER BY t.abbreviation, p.last_name
    """,
    connection,
)

players

## Top Scoring Performances

In [ ]:
top_scoring = pd.read_sql_query(
    """
    SELECT
        p.first_name || ' ' || p.last_name AS player,
        t.abbreviation AS team,
        g.game_date,
        s.points,
        s.rebounds,
        s.assists
    FROM player_game_stats s
    JOIN players p ON s.player_id = p.player_id
    JOIN teams t ON p.team_id = t.team_id
    JOIN games g ON s.game_id = g.game_id
    ORDER BY s.points DESC
    LIMIT 5
    """,
    connection,
)

top_scoring

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(top_scoring["player"], top_scoring["points"], color="steelblue")
plt.title("Top Scoring Performances")
plt.xlabel("Player")
plt.ylabel("Points")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Team Statistics

In [ ]:
team_stats = pd.read_sql_query(
    """
    SELECT
        t.abbreviation AS team,
        ROUND(AVG(s.points), 2) AS average_points,
        ROUND(AVG(s.rebounds), 2) AS average_rebounds,
        ROUND(AVG(s.assists), 2) AS average_assists,
        ROUND(SUM(s.field_goals_made) * 1.0 /
            NULLIF(SUM(s.field_goals_attempted), 0), 3) AS field_goal_pct
    FROM player_game_stats s
    JOIN players p ON s.player_id = p.player_id
    JOIN teams t ON p.team_id = t.team_id
    GROUP BY t.team_id, t.abbreviation
    ORDER BY average_points DESC
    """,
    connection,
)

team_stats

In [ ]:
team_stats.plot(
    x="team",
    y=["average_points", "average_rebounds", "average_assists"],
    kind="bar",
    figsize=(8, 4),
)
plt.title("Average Player Statistics by Team")
plt.xlabel("Team")
plt.ylabel("Average")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Injury Report

In [ ]:
injury_report = pd.read_sql_query(
    """
    SELECT
        p.first_name || ' ' || p.last_name AS player,
        t.abbreviation AS team,
        i.report_date,
        i.status,
        i.description,
        COALESCE(i.expected_return_date, 'TBD') AS expected_return
    FROM injuries i
    JOIN players p ON i.player_id = p.player_id
    JOIN teams t ON p.team_id = t.team_id
    WHERE i.status != 'Available'
    ORDER BY i.report_date DESC
    """,
    connection,
)

injury_report

## Game Results

In [ ]:
game_results = pd.read_sql_query(
    """
    SELECT
        g.game_date,
        away.abbreviation AS away_team,
        g.away_score,
        home.abbreviation AS home_team,
        g.home_score,
        CASE
            WHEN g.home_score > g.away_score THEN home.abbreviation
            ELSE away.abbreviation
        END AS winner
    FROM games g
    JOIN teams home ON g.home_team_id = home.team_id
    JOIN teams away ON g.away_team_id = away.team_id
    ORDER BY g.game_date
    """,
    connection,
)

game_results

## Advanced Metric Leaders

In [ ]:
advanced_metrics = pd.read_sql_query(
    """
    SELECT
        p.first_name || ' ' || p.last_name AS player,
        t.abbreviation AS team,
        m.usage_rate,
        m.true_shooting_pct,
        m.player_efficiency_rating,
        m.offensive_rating,
        m.defensive_rating
    FROM advanced_player_metrics m
    JOIN players p ON m.player_id = p.player_id
    JOIN teams t ON p.team_id = t.team_id
    ORDER BY m.offensive_rating DESC
    """,
    connection,
)

advanced_metrics

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(
    advanced_metrics["player"],
    advanced_metrics["offensive_rating"],
    color="darkgreen",
)
plt.title("Offensive Rating Leaders")
plt.xlabel("Offensive Rating")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Statistical Analysis After SQL Retrieval

The SQL query below retrieves player-game records. Pandas is then used to calculate descriptive statistics and look for relationships between the variables.

In [ ]:
statistical_data = pd.read_sql_query(
    """
    SELECT
        p.first_name || ' ' || p.last_name AS player,
        t.abbreviation AS team,
        s.minutes_played,
        s.points,
        s.rebounds,
        s.assists,
        s.turnovers
    FROM player_game_stats s
    JOIN players p ON s.player_id = p.player_id
    JOIN teams t ON p.team_id = t.team_id
    """,
    connection,
)

statistical_summary = statistical_data[
    ["minutes_played", "points", "rebounds", "assists", "turnovers"]
].describe().round(2)

statistical_summary

In [ ]:
player_summary = statistical_data.groupby("player").agg(
    games=("points", "count"),
    average_points=("points", "mean"),
    points_standard_deviation=("points", "std"),
    average_rebounds=("rebounds", "mean"),
    average_assists=("assists", "mean"),
).round(2)

player_summary.sort_values("average_points", ascending=False)

In [ ]:
correlation_matrix = statistical_data[
    ["minutes_played", "points", "rebounds", "assists", "turnovers"]
].corr().round(3)

correlation_matrix

The statistical module tests two initial hypotheses. The first is that players with more minutes will generally score more points. The second is that the team with the strongest average scoring and field-goal percentage will also perform well in the game results. The prototype sample is too small for a final conclusion, but this step shows how SQL results can be analyzed after retrieval.

## Query Performance

In [ ]:
from time import perf_counter

performance_query = """
SELECT player_id, game_id, points
FROM player_game_stats
ORDER BY points DESC
LIMIT 5
"""

query_plan = pd.read_sql_query(
    "EXPLAIN QUERY PLAN " + performance_query,
    connection,
)

iterations = 1000
start_time = perf_counter()
for _ in range(iterations):
    connection.execute(performance_query).fetchall()
elapsed_ms = (perf_counter() - start_time) * 1000

performance_results = pd.DataFrame(
    {
        "iterations": [iterations],
        "total_ms": [round(elapsed_ms, 3)],
        "average_ms": [round(elapsed_ms / iterations, 5)],
    }
)

display(query_plan)
performance_results

The query plan is used to verify whether SQLite uses the points index instead of sorting the entire table without an index. The timing loop gives a repeatable baseline for the current framework. Because the prototype has only 16 player-game rows, the result is a baseline rather than a scalability claim. The same test will be repeated with a larger data set in the final project.

In [ ]:
connection.close()